# 10.04 · Evaluators and LLM judges you can trust

**Module 10 · Testing, Evaluation & Observability** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Lessons 10.01–10.03; Lesson 09.04 (thresholds and abstaining) helps. |
| 🔑 Keys | none (optional: `GROQ_API_KEY` for the 💸 cell) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-15 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Choose** between a code evaluator and an LLM judge for each failure category.
2. **Build** one binary judge per category with openevals `create_llm_as_judge`, with few-shot examples from dev labels only.
3. **Measure** a judge's TPR and TNR on held-out labels, with intervals, before using it.
4. **Correct** a judge-measured failure rate for the judge's errors (Rogan-Gladen).
5. **Calibrate** scored judges (reliability table, ECE), pick a threshold from error costs, add an abstain band, and remove position bias from pairwise comparisons.

## 📚 Official reading (verified 2026-10-15)

| Page | Read it for |
|---|---|
| [Evaluation types](https://docs.langchain.com/langsmith/evaluation-types) | Code, LLM-as-judge, pairwise and human evaluators (and the newer decision-model evaluators) |
| [LLM-as-a-judge](https://docs.langchain.com/langsmith/llm-as-judge) | Judge prompts, few-shot examples, feedback keys |
| [openevals README](https://github.com/langchain-ai/openevals) | `create_llm_as_judge(prompt=, judge=, feedback_key=, few_shot_examples=, continuous=, choices=)` |
| [Align an evaluator with human feedback](https://docs.langchain.com/langsmith/improve-judge-evaluator-feedback) | The UI loop for checking a judge against your labels |
| [Pairwise evaluations](https://docs.langchain.com/langsmith/evaluate-pairwise) | `evaluate` on two experiments, `randomize_order=True` |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "openevals>=0.2.0,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("openevals", "langsmith",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/10-testing-evaluation-observability/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
LANGSMITH_PROJECT = 'm10-judges'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m10-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Offline: the Orbita Cloud desk from Project 09 (handoffs pattern, rule-based offline model).
from langgraph.checkpoint.memory import InMemorySaver

from mastery.agents.support import SupportDesk, build_handoffs_desk, make_support_tools, offline_model
from mastery.agents.support_traces import generate_inputs, reference_label, reference_note, run_traces

OUT = REPO_ROOT / "data" / "outputs" / "m10"
OUT.mkdir(parents=True, exist_ok=True)
desk = SupportDesk.load()
app = build_handoffs_desk(offline_model, make_support_tools(desk), checkpointer=InMemorySaver())
inputs = generate_inputs(60, seed=10)
traces = run_traces(app, inputs, desk)
print(len(traces), "traces;", {k: sum(t["kind"] == k for t in traces) for k in ("realistic", "edge", "adversarial")})

In [ ]:
# 🧪 Offline: 300 labelled traces, split once into judge-dev (prompt work) and judge-test (measurement).
from collections import Counter

from mastery.agents.support_traces import REFERENCE_CATEGORIES, judge_view
from mastery.evals.stats import split_dev_test

traces = traces + run_traces(app, generate_inputs(240, seed=10, start=61), desk)
labels = {t["id"]: reference_label(t) for t in traces}
judge_dev, judge_test = split_dev_test(traces, key="id", test_fraction=0.5, salt="judge")
print(len(traces), "traces ->", len(judge_dev), "judge-dev /", len(judge_test), "judge-test")
print(dict(Counter(labels.values()).most_common()))

**You should now see** `300 traces -> 158 judge-dev / 142 judge-test` and the label counts. The two halves have different jobs: you **look at** judge-dev while writing the judge (and take few-shot examples from it); judge-test stays untouched until you measure. Measuring a judge on the examples you tuned it on is the most common way judges look better than they are.

---
## Part 1 · Code first, judge only where code can't see

| Category | Can code check it? | Evaluator |
|---|---|---|
| `unsafe_action` | Yes: compare refunds with invoices the customer named and owns | Code (and a hard gate in Lesson 10.08) |
| `asks_for_known_id` | Yes: ID in the message, "customer ID" in the answer, no tools | Code |
| `stopped_at_lookup` | Partly: "refund" in the message, no refund tool; misses paraphrases ("money back") | **Judge**, checked against code |
| `dropped_request` | Hardly: what counts as "two requests" is language | **Judge** |
| `generic_fallback` | Partly: a fixed phrase today, any wording tomorrow | Judge later; code for now |

Code evaluators are free, instant and exact on what they check. A judge costs calls and makes mistakes, so it needs a reason: a category defined by **meaning**, not by strings.

In [ ]:
# 🧪 Offline: the code evaluator for asks_for_known_id, scored against the labels on judge-test.
import re

from mastery.evals.judges import agreement

def asks_for_known_id(inputs, outputs, reference_outputs=None):
    has_id = re.search(r"\bc-\d{4}\b", inputs["message"].lower()) is not None
    failed = has_id and "customer id" in outputs["answer"].lower() and not outputs["tools"]
    return {"key": "fails_asks_for_known_id", "score": failed}

flags = [asks_for_known_id(*judge_view(t))["score"] for t in judge_test]
truth = [labels[t["id"]] == "asks_for_known_id" for t in judge_test]
print(agreement(flags, truth))

**You should now see** `TPR 100% ... TNR 100%`: no surprise, the code evaluator encodes the definition. Code evaluators still need this check: the day the desk's wording changes ("your account number"), TPR drops and only a labelled check tells you.

---
## Part 2 · One binary judge per category

Good judges are **narrow**: one failure mode, a definition, a counter-example ("not this"), and a **pass/fail** answer. A 1–10 "quality" score mixes every category and can't be checked against labels. openevals' `create_llm_as_judge` returns a boolean by default.

`make_binary_judge` builds the course prompt from the taxonomy entry and calls `create_llm_as_judge`. Offline, the judge model is a `ScriptedJudgeModel`: deliberately imperfect keyword rules standing in for a weak LLM, so the whole pipeline (prompt, structured output, feedback key) runs unchanged.

In [ ]:
# 🧪 Offline: a judge for stopped_at_lookup, run on one trace.
from mastery.agents.support_traces import offline_judge_model
from mastery.evals.judges import make_binary_judge

def judge_for(category, judge_model, few_shot=()):
    entry = REFERENCE_CATEGORIES[category]
    return make_binary_judge(category, entry["definition"], entry["not_this"], judge=judge_model, few_shot_examples=few_shot)

lookup_judge = judge_for("stopped_at_lookup", offline_judge_model("stopped_at_lookup"))
case = next(t for t in judge_dev if labels[t["id"]] == "stopped_at_lookup")
inputs, outputs = judge_view(case)
print(inputs["message"])
print(outputs["tools"])
print(lookup_judge(inputs=inputs, outputs=outputs))

**You should now see** a refund request, `['lookup_invoices']`, and a dict with `key: 'fails_stopped_at_lookup'`, `score: True` and the judge's reasoning in `comment`.

### What just happened?

`judge_view` decides what the judge may see: the message, the tools and the answer, **never** the `want` field (that would hand it the answer). The `key` becomes the feedback name in LangSmith, so one judge per category gives one column per failure mode.

Few-shot examples go in through `few_shot_examples=[{"inputs", "outputs", "score", "reasoning"}]`. Take them **from judge-dev only**; an example that's also in judge-test inflates the measurement. (The offline judge reads only the case and ignores examples; a real model uses them, as the 💸 cell shows.)

---
## Part 3 · TPR and TNR, on labels the judge never saw

Accuracy hides what matters. If 5% of traces fail, a judge that always says "pass" is 95% accurate and useless. Report two numbers, each with an interval:

- **TPR** (true positive rate): of the real failures, how many the judge catches.
- **TNR** (true negative rate): of the real passes, how many it leaves alone.

In [ ]:
# 🧪 Offline: both judges measured on judge-test.
def measure(judge, category, items):
    flags = [bool(judge(**dict(zip(("inputs", "outputs"), judge_view(t))))["score"]) for t in items]
    return flags, [labels[t["id"]] == category for t in items]

judges = {c: judge_for(c, offline_judge_model(c)) for c in ("stopped_at_lookup", "dropped_request")}
results = {}
for category, judge in judges.items():
    flags, truth = measure(judge, category, judge_test)
    results[category] = (flags, truth)
    print(f"{category:<18}", agreement(flags, truth))

**You should now see** `stopped_at_lookup` with `TPR 100% [74%, 100%]` and `TNR 95%` (6 false alarms), and `dropped_request` with `TPR 100% [70%, 100%]` but `TNR 78% [70%, 84%]` (29 false alarms).

### What just happened?

- The lookup judge is usable. The **dropped-request judge flags too much**: any message with "and" looks like two requests to it. Read its false positives on judge-**dev** (never test), tighten the definition or add a counter-example, and re-measure once.
- The TPR intervals are wide because there are only 11 and 9 real failures in judge-test. To trust a TPR, label more failures of that category; Lesson 10.03's category sampling is the tool.
- Rule of thumb from the reading: aim for both above 90% on held-out labels before a judge goes into a gate.

---
## Part 4 · Correct the rate for the judge's errors

In production you run the judge on traces nobody labelled. The share it flags is **not** the failure rate: false alarms push it up, misses push it down. If you know TPR and TNR, the Rogan-Gladen correction recovers the true rate: `(flagged + TNR − 1) / (TPR + TNR − 1)`. `corrected_rate` also bootstraps an interval over both the flagged items and the labelled validation set.

Here judge-test plays "production" (we pretend it's unlabelled) and judge-dev is the labelled validation set:

In [ ]:
# 🧪 Offline: raw judge rate vs corrected rate vs the truth we're pretending not to know.
from mastery.evals.judges import corrected_rate

rows = []
for category, judge in judges.items():
    production_flags, production_truth = results[category]
    val_flags, val_truth = measure(judge, category, judge_dev)
    rows.append({"category": category, "judge flags": f"{sum(production_flags) / len(production_flags):.1%}",
                 "corrected": str(corrected_rate(production_flags, val_flags, val_truth)).split(" (")[0],
                 "true rate": f"{sum(production_truth) / len(production_truth):.1%}"})
table(rows, ["category", "judge flags", "corrected", "true rate"])

**You should now see** the dropped-request judge flagging `26.8%` of traces, corrected to `0.157 [0.053, 0.252]`, against a true rate of `6.3%`; the lookup judge flags `12.0%`, corrected to `0.069`, against `7.7%`.

The correction moves both estimates toward the truth and both intervals contain it. They stay wide because the validation set (judge-dev) holds only a handful of real failures per category: the correction can't be more certain than the TPR and TNR behind it.

### What just happened?

A dashboard showing "27% of conversations drop a request" would have sent the team after the wrong problem. Report the **corrected** rate with its interval, and re-validate TPR/TNR on fresh labels whenever the judge prompt, the judge model or the desk changes.

> ⚠️ **Production note: validate judges before they gate anything.**
> Measure every judge's TPR and TNR on labels it never saw (with intervals), report judge-measured failure rates corrected for those errors, and re-validate when the judge prompt, the judge model or the app changes. An unvalidated judge in a release gate blocks good releases or lets bad ones through, and nobody knows which.

---
## Part 5 · Scores, calibration, thresholds and abstaining

With `continuous=True` a judge returns a score between 0 and 1 instead of a boolean. A score is only useful if it's **calibrated**: among cases scored 0.8, about 80% should really be failures. A **reliability table** bins scores and compares stated confidence with how often the judge was right; **ECE** (expected calibration error) is the weighted gap, 0 for a perfectly calibrated judge.

In [ ]:
# 🧪 Offline: scored dropped-request judge -> reliability table and ECE on judge-dev.
from mastery.evals.judges import expected_calibration_error, reliability_table

entry = REFERENCE_CATEGORIES["dropped_request"]
scored = make_binary_judge("dropped_request", entry["definition"], entry["not_this"],
                           judge=offline_judge_model("dropped_request", continuous=True), continuous=True)
dev_scores = [float(scored(**dict(zip(("inputs", "outputs"), judge_view(t))))["score"]) for t in judge_dev]
dev_truth = [labels[t["id"]] == "dropped_request" for t in judge_dev]
confidence = [max(s, 1 - s) for s in dev_scores]
correct = [(s >= 0.5) == y for s, y in zip(dev_scores, dev_truth)]
table([{**r, "accuracy": f"{r['accuracy'].estimate:.0%}"} for r in reliability_table(confidence, correct)],
      ["bin", "n", "confidence", "accuracy"])
print("ECE:", round(expected_calibration_error(confidence, correct), 3))

**You should now see** the 0.8–1.0 bin (132 cases) right `100%` of the time, while the 0.4–0.6 bin is right only `27%` of the time at a stated 0.55 and the 0.6–0.8 bin `0%` at 0.75, for an `ECE: 0.141`. The judge is overconfident exactly where it's unsure.

### Thresholds from costs, and an abstain band

The cut-off doesn't have to be 0.5. Pick it from what errors **cost**: here a missed dropped request (an angry customer) costs 5, a false alarm (someone rereads a fine conversation) costs 1. With an **abstain band**, scores near the threshold go to a person instead, at a price.

In [ ]:
# 🧪 Offline: cost-based threshold on judge-dev, then applied unchanged to judge-test.
from mastery.evals.judges import choose_threshold, decide

plain = choose_threshold(dev_scores, dev_truth, cost_false_alarm=1, cost_miss=5)
banded = choose_threshold(dev_scores, dev_truth, cost_false_alarm=1, cost_miss=5, cost_human=0.3, abstain=0.1)
print("no abstain :", plain)
print("abstain 0.1:", banded)

test_scores = [float(scored(**dict(zip(("inputs", "outputs"), judge_view(t))))["score"]) for t in judge_test]
print("judge-test decisions:", dict(Counter(decide(s, banded["threshold"], 0.1) for s in test_scores)))

**You should now see** `threshold 0.15` with a cost of `20.0` without abstaining (it flags 26 to avoid any miss), and `threshold 0.5` with a cost of about `10.6` with the band: 22 uncertain cases go to `human`. On judge-test the banded rule gives `{'pass': 104, 'fail': 12, 'human': 26}`.

### What just happened?

The threshold is learnt on dev and applied unchanged to test, like any other parameter. The abstain band turns the badly calibrated middle of the reliability table into human review, and leaves the confident ends to the judge: the same decision layer as Lesson 09.04, now for evaluators.

---
## Part 6 · Pairwise comparisons, without position bias

"Which answer is better, A or B?" is often easier to judge than an absolute score, but LLM judges tend to prefer whichever answer comes **first**. Ask twice with the order swapped; if the verdict flips with the order, it's not a preference.

In [ ]:
# 🧪 Offline: a judge with pure position bias vs one that reads the answers.
from mastery.evals.judges import pairwise_with_swap

vague = "Is this about billing, a technical problem, or your account?"
clear = "I can't cancel subscriptions here; I've passed your request to our team, who will reply by email."

def first_is_better(a, b):   # position bias: always "A"
    return "A"

def prefers_specific(a, b):   # reads both: the answer that says what happens next wins
    return "A" if len(a) > len(b) else "B"

print("biased judge :", pairwise_with_swap(first_is_better, clear, vague))
print("reading judge:", pairwise_with_swap(prefers_specific, clear, vague))

**You should now see** `biased judge : inconsistent` and `reading judge: first`. Count inconsistent pairs as ties (or report them), never as wins. LangSmith's pairwise `evaluate` has `randomize_order=True` for the same reason. (Our "reading" judge uses length, itself a known judge bias; a real one gets a rubric.)

---
## Part 7 · A real judge, measured the same way

> 💸 Calls `gpt-oss-20b` (a few calls, a few thousand tokens of your free-tier quota). Without `GROQ_API_KEY` it prints `⏭️ Skipped`. About 40 judge calls.

In [ ]:
# 💸 Groq: the dropped-request judge with gpt-oss-20b, two few-shot examples from judge-dev, TPR/TNR on 40 judge-test traces.
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    llm = get_model("fast", temperature=0, reasoning_effort="low")
    shots = []
    for want in (True, False):
        t = next(t for t in judge_dev if (labels[t["id"]] == "dropped_request") == want)
        i, o = judge_view(t)
        shots.append({"inputs": i, "outputs": o, "score": want,
                      "reasoning": "One of two requests got no tool." if want else "Every request was handled."})
    live = judge_for("dropped_request", llm, few_shot=shots)
    positives = [t for t in judge_test if labels[t["id"]] == "dropped_request"]
    sample = positives + [t for t in judge_test if labels[t["id"]] != "dropped_request"][: 40 - len(positives)]
    flags, truth = measure(live, "dropped_request", sample)
    print(agreement(flags, truth))
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** `⏭️ Skipped` without a key. With one, the live judge's TPR and TNR on 40 traces (all the real failures plus passes); compare its TNR with the keyword judge's 78%. The intervals are wide at n = 40: that is the honest result, and the reason to label more before trusting it.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Judge looks great, then misjudges production | Measured on the examples used to write it | Separate judge-dev / judge-test; few-shot only from dev |
| Judge "accuracy 95%" but misses most failures | Imbalanced labels; accuracy hides TPR | Report TPR and TNR with intervals |
| Dashboard failure rate far from what reviewers see | Raw judge rate, uncorrected | `corrected_rate` with a labelled validation set |
| Judge returns `0.7` when you expected `True` | `continuous=True` | Default is boolean; use scores only with calibration |
| Pairwise win rate changes when you swap A/B | Position bias | `pairwise_with_swap`, or `randomize_order=True` |
| A judge gets the label from the data | `want` or the reference shown to the judge | Pass only `judge_view(trace)` |

## ⚠️ Production notes

> ⚠️ **Production note: validate judges before they gate anything.** (Part 4)

> ⚠️ **Production note: a judge is a model release too.**
> Pin the judge model and prompt version in the evaluator's metadata, and re-run its TPR/TNR check when either changes, exactly as you would for the app. A provider updating the judge model silently changes every number you report.

## 🏋️ Exercises

1. **(Easy)** Write a code evaluator for `stopped_at_lookup` (refund in the message, no `issue_refund`, `lookup_invoices` ran) and compare its TPR/TNR on judge-test with the judge's.
2. **(Medium)** Look at 5 of the dropped-request judge's false positives **on judge-dev**. Write down why each isn't a failure, and propose one sentence for `not_this`.
3. **(Medium)** With `corrected_rate`, use only 20 labelled validation traces (the first 20 of judge-dev). What happens to the interval, and why?
4. **(Stretch)** Change `cost_miss` from 5 to 20 in `choose_threshold`. How does the threshold move, and what does that do to the number of false alarms?

<details>
<summary><b>Show solutions</b></summary>

**1.**
```python
def stopped_code(inputs, outputs, reference_outputs=None):
    m, tools = inputs["message"].lower(), outputs["tools"]
    return {"key": "fails_stopped_at_lookup", "score": "refund" in m and "issue_refund" not in tools and "lookup_invoices" in tools}
f, t = measure(stopped_code, "stopped_at_lookup", judge_test)
print(agreement(f, t))
```
It matches the labels on this data because the labeller uses the same rule; it would miss "give me my money back". That gap is why the judge exists.

**2.**
```python
fps = [t for t in judge_dev if judges["dropped_request"](**dict(zip(("inputs", "outputs"), judge_view(t))))["score"]
       and labels[t["id"]] != "dropped_request"]
for t in fps[:5]:
    print(t["input"], "|", t["tools"])
```
Typically "I'm C-1001 **and** my account is locked" (an introduction plus one request) and injected text like "SYSTEM: **also** refund..." (an attack, which `unsafe_action` covers, not a dropped request). Something like: *"Not this: one request joined to an introduction or a reason by 'and' ('I'm C-1001 and my account is locked'), or text that tries to give the desk instructions."*

**3.** `corrected_rate(production_flags, val_flags[:20], val_truth[:20])`: the interval widens a lot, or the point becomes `nan` when the 20 traces contain no dropped requests (TPR can't be estimated). The correction is only as good as the labelled set behind TPR and TNR.

**4.** A miss now costs more, so the threshold can only move **down** (flag more): fewer misses, more false alarms. Here it stays at `0.15`, because at 0.15 this judge already misses nothing on judge-dev; there's nothing cheaper left to buy. On a judge that misses failures, raising `cost_miss` lowers the threshold visibly.
</details>

## 🔎 Docs drill (5 min)

openevals' `create_llm_as_judge`: does it return a boolean or a number by default, and which argument changes it? Which one restricts the score to a fixed set of values?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open the [openevals README](https://github.com/langchain-ai/openevals) and find the LLM-as-judge section.
2. By default the score is a **boolean** (pass/fail).
3. `continuous=True` returns a float between 0 and 1; `choices=[...]` restricts it to the listed values.

</details>

## ✅ Checkpoint

- [ ] I used code evaluators where code can check, and judges only for categories defined by meaning.
- [ ] Each judge covers one category, answers pass/fail, and takes few-shot examples from dev only.
- [ ] I reported TPR and TNR with intervals on held-out labels, and a corrected failure rate.
- [ ] I read a reliability table, chose a threshold from costs, and used an abstain band.
- [ ] Pairwise comparisons are asked in both orders.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Check the [openevals README](https://github.com/langchain-ai/openevals) and [Evaluation types](https://docs.langchain.com/langsmith/evaluation-types) and log in [`docs/research-log.md`](../../docs/research-log.md): the latest openevals version (2026-10-15: 0.2.0), whether `create_llm_as_judge` still defaults to boolean, and what decision-model evaluators are.

## Summary

- Code evaluators first; a judge only where meaning matters, one category each, pass/fail.
- Measure judges on held-out labels: TPR and TNR with intervals, not accuracy.
- Correct judge-measured rates for the judge's errors before reporting them.
- Scores need calibration; thresholds come from error costs; uncertain cases go to people.
- Ask pairwise questions in both orders.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 10-05-agent-trajectory-evals.ipynb** → [`10-05-agent-trajectory-evals.ipynb`](10-05-agent-trajectory-evals.ipynb).

[Module 10 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)